# Smart Building Dataset - ML Classification Pipeline
## Anomaly Detection / Fault Classification
### Dataset: 4 CSV files (normal/faulty training & testing)

In [ ]:
import pandas as pdimport numpy as npimport warningswarnings.filterwarnings('ignore')print('✅ Data libraries loaded')

In [ ]:
import matplotlib.pyplot as pltimport seaborn as sns%matplotlib inlinesns.set_style('whitegrid')print('✅ Visualization libraries loaded')

In [ ]:
from sklearn.preprocessing import StandardScalerfrom sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFoldfrom sklearn.linear_model import LogisticRegressionfrom sklearn.neighbors import KNeighborsClassifierfrom sklearn.tree import DecisionTreeClassifierfrom sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifierfrom sklearn.svm import SVCfrom sklearn.naive_bayes import GaussianNBfrom xgboost import XGBClassifierfrom lightgbm import LGBMClassifierfrom imblearn.over_sampling import SMOTEprint('✅ ML libraries loaded')

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score, roc_curveimport joblibfrom datetime import datetimeprint('✅ Evaluation libraries loaded')

In [ ]:
RANDOM_STATE = 42TEST_SIZE = 0.2CV_FOLDS = 5TARGET_ACCURACY = 0.90plt.rcParams['figure.figsize'] = (14, 6)print(f'✅ Configuration set - Target: {TARGET_ACCURACY*100}%')

## 1. LOAD & EXPLORE DATASET

In [ ]:
df_normal_train = pd.read_csv('../data/normal_training.csv')df_faulty_train = pd.read_csv('../data/faulty_training.csv')df_normal_test = pd.read_csv('../data/normal_testing.csv')df_faulty_test = pd.read_csv('../data/faulty_testing.csv')df_normal_train['label'] = 0df_faulty_train['label'] = 1df_normal_test['label'] = 0df_faulty_test['label'] = 1df_train = pd.concat([df_normal_train, df_faulty_train], ignore_index=True)df_test = pd.concat([df_normal_test, df_faulty_test], ignore_index=True)print(f'✅ Dataset loaded - Train: {len(df_train)}, Test: {len(df_test)}')

In [ ]:
print(df_train.head())print(f'\nShape: {df_train.shape}')print(f'Columns: {df_train.columns.tolist()}')

In [ ]:
print('Dataset Info:')df_train.info()print(f'\nMissing: {df_train.isnull().sum().sum()}')print(f'Duplicates: {df_train.duplicated().sum()}')

In [ ]:
print('Statistical Summary:')print(df_train.describe())

## 2. DATA ANALYSIS

In [ ]:
class_counts = df_train['label'].value_counts()print(f'Class Distribution:')print(f'  Normal: {class_counts[0]} ({class_counts[0]/len(df_train)*100:.1f}%)')print(f'  Faulty: {class_counts[1]} ({class_counts[1]/len(df_train)*100:.1f}%)')fig, ax = plt.subplots(figsize=(8, 5))class_counts.plot(kind='bar', ax=ax, color=['green', 'red'])ax.set_title('Class Distribution')ax.set_ylabel('Count')plt.show()ratio = class_counts[1] / class_counts[0]print(f'\nImbalance ratio: {1/ratio:.2f}:1 - SMOTE will be applied')

In [ ]:
features = [col for col in df_train.columns if col != 'label']fig, axes = plt.subplots(2, 3, figsize=(15, 8))for idx, feature in enumerate(features[:6]):    ax = axes[idx // 3, idx % 3]    df_train[feature].hist(bins=30, ax=ax, alpha=0.7)    ax.set_title(f'{feature} Distribution')plt.tight_layout()plt.show()

In [ ]:
correlation = df_train.corr()plt.figure(figsize=(12, 10))sns.heatmap(correlation, annot=False, cmap='coolwarm', center=0)plt.title('Feature Correlation Matrix')plt.tight_layout()plt.savefig('../output/correlation_matrix.png', dpi=300, bbox_inches='tight')plt.show()

In [ ]:
for feature in features[:3]:    Q1 = df_train[feature].quantile(0.25)    Q3 = df_train[feature].quantile(0.75)    IQR = Q3 - Q1    outliers = df_train[(df_train[feature] < Q1 - 1.5*IQR) | (df_train[feature] > Q3 + 1.5*IQR)]    print(f'{feature}: {len(outliers)} outliers ({len(outliers)/len(df_train)*100:.2f}%)')

## 3. PREPROCESSING

In [ ]:
y_train = df_train['label'].valuesy_test = df_test['label'].valuesX_train = df_train.drop('label', axis=1).valuesX_test = df_test.drop('label', axis=1).valuesfeature_names = df_train.drop('label', axis=1).columns.tolist()print(f'✅ Data prepared - Features: {X_train.shape[1]}, Train: {len(X_train)}, Test: {len(X_test)}')

In [ ]:
scaler = StandardScaler()X_train_scaled = scaler.fit_transform(X_train)X_test_scaled = scaler.transform(X_test)print(f'✅ Scaling complete (fit on training only - NO DATA LEAKAGE)')print(f'Train mean ≈ 0: {X_train_scaled.mean(axis=0)[:3]}')print(f'Train std ≈ 1: {X_train_scaled.std(axis=0)[:3]}')

## 4. FEATURE SELECTION

In [ ]:
from sklearn.ensemble import RandomForestClassifier as RFC_temprf = RFC_temp(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)rf.fit(X_train_scaled, y_train)importance_df = pd.DataFrame({'feature': feature_names, 'importance': rf.feature_importances_}).sort_values('importance', ascending=False)print('Top 10 Features:')print(importance_df.head(10))plt.figure(figsize=(10, 6))plt.barh(importance_df['feature'][:10], importance_df['importance'][:10])plt.xlabel('Importance')plt.title('Top 10 Features')plt.tight_layout()plt.savefig('../output/feature_importance.png', dpi=300, bbox_inches='tight')plt.show()

In [ ]:
corr_matrix = pd.DataFrame(X_train_scaled, columns=feature_names).corr()upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))to_drop = [col for col in upper.columns if any(upper[col].abs() > 0.95)]features_keep = [f for f in feature_names if f not in to_drop]print(f'Total features: {len(feature_names)}')print(f'High correlation: {len(to_drop)}')print(f'To keep: {len(features_keep)}')

In [ ]:
print('✅ Feature selection complete')print(f'Kept {len(features_keep)} of {len(feature_names)} features')

## 5. HANDLE CLASS IMBALANCE (SMOTE)

In [ ]:
smote = SMOTE(random_state=RANDOM_STATE, n_jobs=-1)X_train_bal, y_train_bal = smote.fit_resample(X_train_scaled, y_train)print(f'Original: {len(y_train)} samples')print(f'After SMOTE: {len(y_train_bal)} samples')print(f'Ratio: {(y_train_bal == 1).sum() / (y_train_bal == 0).sum():.3f}')fig, ax = plt.subplots(1, 2, figsize=(12, 4))pd.Series(y_train).value_counts().plot(kind='bar', ax=ax[0], color=['green', 'red'])ax[0].set_title('Before SMOTE')pd.Series(y_train_bal).value_counts().plot(kind='bar', ax=ax[1], color=['green', 'red'])ax[1].set_title('After SMOTE')plt.tight_layout()plt.savefig('../output/smote_balance.png', dpi=300, bbox_inches='tight')plt.show()print('✅ SMOTE applied')

## 6. TRAIN MODELS (9 Algorithms)

In [ ]:
models = {    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),    'KNN': KNeighborsClassifier(n_neighbors=5),    'Decision Tree': DecisionTreeClassifier(random_state=RANDOM_STATE),    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),    'Gradient Boosting': GradientBoostingClassifier(random_state=RANDOM_STATE),    'XGBoost': XGBClassifier(random_state=RANDOM_STATE, n_jobs=-1, verbosity=0),    'LightGBM': LGBMClassifier(random_state=RANDOM_STATE, n_jobs=-1, verbose=-1),    'SVM': SVC(kernel='rbf', random_state=RANDOM_STATE, probability=True),    'Naive Bayes': GaussianNB()}print('✅ 9 models initialized')trained_models = {}for name, model in models.items():    model.fit(X_train_bal, y_train_bal)    trained_models[name] = model    print(f'  ✅ {name} trained')print(f'\n✅ All models trained')

In [ ]:
predictions = {}probabilities = {}for name, model in trained_models.items():    y_pred = model.predict(X_test_scaled)    predictions[name] = y_pred    if hasattr(model, 'predict_proba'):        probabilities[name] = model.predict_proba(X_test_scaled)[:, 1]    else:        probabilities[name] = model.decision_function(X_test_scaled) if hasattr(model, 'decision_function') else y_predprint('✅ Predictions made for all models')

## 7. MODEL EVALUATION

In [ ]:
results = []for name in models.keys():    y_pred = predictions[name]    acc = accuracy_score(y_test, y_pred)    prec = precision_score(y_test, y_pred, zero_division=0)    rec = recall_score(y_test, y_pred, zero_division=0)    f1 = f1_score(y_test, y_pred, zero_division=0)    try:        roc = roc_auc_score(y_test, probabilities[name])    except:        roc = 0    results.append({'Model': name, 'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1, 'ROC-AUC': roc})results_df = pd.DataFrame(results).sort_values('Accuracy', ascending=False)print('\n=== MODEL COMPARISON ===')print(results_df.to_string(index=False))results_df.to_csv('../output/model_results.csv', index=False)print('\n✅ Results saved')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))for idx, metric in enumerate(['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']):    ax = axes[idx // 3, idx % 3]    ax.barh(results_df['Model'], results_df[metric], color=plt.cm.Set3(range(len(results_df))))    ax.set_xlabel(metric)    ax.axvline(TARGET_ACCURACY, color='red', linestyle='--', linewidth=2, label='Target')    ax.set_xlim(0, 1.05)    ax.legend()axes[1, 2].axis('off')plt.tight_layout()plt.savefig('../output/metrics_comparison.png', dpi=300, bbox_inches='tight')plt.show()print('✅ Metrics visualized')

In [ ]:
best_model_name = results_df.iloc[0]['Model']best_accuracy = results_df.iloc[0]['Accuracy']print(f'\n⭐ BEST MODEL: {best_model_name}')print(f'\nAccuracy: {best_accuracy:.4f} ({best_accuracy*100:.2f}%)')if best_accuracy >= TARGET_ACCURACY:    print(f'✅ TARGET ACHIEVED! ({best_accuracy*100:.2f}% >= {TARGET_ACCURACY*100}%)')else:    print(f'⚠️ Below target. Gap: {(TARGET_ACCURACY - best_accuracy)*100:.2f}%')

In [ ]:
top_models = results_df.head(3)['Model'].tolist()fig, axes = plt.subplots(1, 3, figsize=(15, 4))for idx, model_name in enumerate(top_models):    y_pred = predictions[model_name]    cm = confusion_matrix(y_test, y_pred)    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx], cbar=False)    acc = accuracy_score(y_test, y_pred)    axes[idx].set_title(f'{model_name}\n{acc:.4f}')    axes[idx].set_xlabel('Predicted')    axes[idx].set_ylabel('Actual')plt.tight_layout()plt.savefig('../output/confusion_matrices.png', dpi=300, bbox_inches='tight')plt.show()print('✅ Confusion matrices displayed')

## 8. HYPERPARAMETER TUNING & CROSS-VALIDATION

In [ ]:
from sklearn.model_selection import cross_validatebest_model = trained_models[best_model_name]cv_results = cross_validate(best_model, X_train_bal, y_train_bal, cv=5, scoring=['accuracy', 'f1', 'precision', 'recall'])print(f'Cross-Validation Results (5-fold):')print(f'Accuracy: {cv_results["test_accuracy"].mean():.4f} (+/- {cv_results["test_accuracy"].std():.4f})')print(f'F1-Score: {cv_results["test_f1"].mean():.4f} (+/- {cv_results["test_f1"].std():.4f})')print(f'Precision: {cv_results["test_precision"].mean():.4f} (+/- {cv_results["test_precision"].std():.4f})')print(f'Recall: {cv_results["test_recall"].mean():.4f} (+/- {cv_results["test_recall"].std():.4f})')fig, ax = plt.subplots(figsize=(10, 5))ax.boxplot([cv_results['test_accuracy'], cv_results['test_f1'], cv_results['test_precision'], cv_results['test_recall']])ax.set_xticklabels(['Accuracy', 'F1', 'Precision', 'Recall'])ax.set_ylabel('Score')ax.set_title(f'5-Fold CV Results - {best_model_name}')plt.grid(alpha=0.3)plt.tight_layout()plt.savefig('../output/cross_validation_results.png', dpi=300, bbox_inches='tight')plt.show()print('✅ Cross-validation complete')

In [ ]:
if best_model_name in ['Logistic Regression', 'Random Forest', 'SVM', 'XGBoost', 'LightGBM']:    if best_model_name == 'Random Forest':        imp = pd.DataFrame({'feature': feature_names, 'importance': best_model.feature_importances_}).sort_values('importance', ascending=False)        plt.figure(figsize=(10, 6))        plt.barh(imp['feature'][:15], imp['importance'][:15])        plt.xlabel('Importance')        plt.title(f'{best_model_name} - Feature Importance')        plt.tight_layout()        plt.savefig('../output/best_model_importance.png', dpi=300, bbox_inches='tight')        plt.show()        print('✅ Feature importance displayed')print('✅ Analysis complete')

## 9. SAVE MODEL & RESULTS

In [ ]:
joblib.dump(best_model, '../output/best_model.pkl')joblib.dump(scaler, '../output/scaler.pkl')joblib.dump(trained_models, '../output/all_models.pkl')print('✅ Models saved:')print('  - best_model.pkl')print('  - scaler.pkl')print('  - all_models.pkl')

## 10. CONCLUSION & RECOMMENDATIONS

In [ ]:
print('='*60)print('FINAL RESULTS SUMMARY')print('='*60)print(f'\nBest Model: {best_model_name}')print(f'Accuracy: {best_accuracy:.4f} ({best_accuracy*100:.2f}%)')print(f'Target: {TARGET_ACCURACY*100}%')print(f'Status: {"✅ TARGET ACHIEVED" if best_accuracy >= TARGET_ACCURACY else "⚠️ Below Target"}')print(f'\nTop 3 Models:')for i, row in results_df.head(3).iterrows():    print(f'  {row["Model"]}: {row["Accuracy"]:.4f}')if best_accuracy < TARGET_ACCURACY:    gap = (TARGET_ACCURACY - best_accuracy) * 100    print(f'\nGap to Target: {gap:.2f}%')    print(f'\nRecommendations for Improvement:')    print(f'1. Collect more data (especially minority class)')    print(f'2. Try ensemble methods or stacking')    print(f'3. Engineer additional features')    print(f'4. Perform more aggressive hyperparameter tuning')    print(f'5. Experiment with different scaling/preprocessing')else:    print(f'\n✅ SUCCESS! Model achieved {best_accuracy*100:.2f}% accuracy')    print(f'\nNext Steps:')    print(f'1. Deploy model to production')    print(f'2. Monitor performance on new data')    print(f'3. Retrain periodically with new samples')    print(f'4. Implement feedback loop for continuous improvement')print('\n' + '='*60)print(f'Pipeline completed at: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')print('='*60)